In [3]:
import requests
import pandas as pd
import numpy as np
import time
import os
from cache import* 
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("ALPHAVANTAGE_API_KEY")

In [4]:
import requests


keywords = 'IBM'
url = f'https://www.alphavantage.co/query?function=SYMBOL_SEARCH&keywords={keywords}&apikey={api_key}'
r = requests.get(url)
print(r.json())

{'bestMatches': [{'1. symbol': 'IBM', '2. name': 'International Business Machines Corp', '3. type': 'Equity', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '1.0000'}, {'1. symbol': 'IBMN', '2. name': 'ISHARES IBONDS DEC 2025 TERM MUNI BOND ETF ', '3. type': 'ETF', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '0.8571'}, {'1. symbol': 'IBMO', '2. name': 'ISHARES IBONDS DEC 2026 TERM MUNI BOND ETF ', '3. type': 'ETF', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '0.8571'}, {'1. symbol': 'IBMP', '2. name': 'ISHARES IBONDS DEC 2027 TERM MUNI BOND ETF ', '3. type': 'ETF', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency':

In [ ]:
# GET ANNUALIZED VOLATILITY
import requests
import pandas as pd
import numpy as np
import time
symbols = ['SWDA.LON']

for symbol in symbols:
    print(f"Processing {symbol}...")

    url = (
        f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY'
        f'&symbol={symbol}&apikey={api_key}&outputsize=full&datatype=csv'
    )

    df = pd.read_csv(url, parse_dates=['timestamp'])
    df = df.sort_values('timestamp')  # oldest to newest

    # Use last 252 trading days for 1-year volatility
    df_last_year = df.tail(252)
    prices = df_last_year['close']
    
    # Convert GBX to GBP if needed
    prices_gbp = prices / 100.0

    # Calculate log returns
    returns = np.log(prices_gbp / prices_gbp.shift(1)).dropna()

    # Daily volatility
    daily_vol = returns.std()

    # Annualized volatility
    annual_vol = daily_vol * np.sqrt(252)

    print(f"{symbol} 1Y annualized volatility: {annual_vol:.2%} (GBP-based)")
    time.sleep(3)


Processing SWDA.LON...
SWDA.LON 1Y annualized volatility: 14.81% (GBP-based)


In [ ]:

# GET ANNUALIZED VOLATILITY IN CHF
import requests
import pandas as pd
import numpy as np
import time
import os
import cache.py
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("ALPHAVANTAGE_API_KEY")
symbols = ['SWDA.LON']

symbol = "SWDA.LON"
url = (
        f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY'
        f'&symbol={symbol}&apikey={api_key}&outputsize=full&datatype=csv'
    )

asset_df = fetch_csv_cached(url=url, function="time_series_daily", key=symbol, ttl_hours=1000, expected_cols=["timestamp", "open", "high", "low", "close", "volume"])



print(asset_df.head())


for symbol in symbols:
    print(f"Processing {symbol}...")

    url = (
        f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY'
        f'&symbol={symbol}&apikey={api_key}&outputsize=full&datatype=csv'
    )
    asset_df=pd.read_csv(url)
    print(asset_df.head())

    # get the asset price data
    asset_df = pd.read_csv(url, parse_dates=['timestamp']).sort_values('timestamp').set_index('timestamp') 
    # Use last 252 trading days for 1-year volatility
    asset_df = asset_df.rename(columns={'close': 'asset_close_GBx'})

    fx_url = (
    f'https://www.alphavantage.co/query?function=FX_DAILY'
    f'&from_symbol=GBP&to_symbol=CHF&apikey={api_key}&outputsize=full&datatype=csv'
    )
    fx_df = pd.read_csv(fx_url, parse_dates=['timestamp']).sort_values('timestamp').set_index('timestamp')
    fx_rate = fx_df['close'].rename( 'fx_rate')


    # Now merge with your asset price series on date and multiply:
    merged = asset_df.join(fx_rate, how='inner')
    merged['close_chf'] = merged['asset_close_GBx'] *merged['fx_rate']  # Convert to CHF

    # Continue as before, but on 'close_chf'
    prices = merged['close_chf'].tail(252)  # Last 252 trading days
    returns = np.log(prices / prices.shift(1)).dropna()


    # Daily volatility
    daily_vol = returns.std()

    # Annualized volatility
    annual_vol = daily_vol * np.sqrt(252)

    print(f"{symbol} 1Y annualized volatility: {annual_vol:.2%} (GHF-based)")
    # time.sleep(3)


In [ ]:
import pandas as pd, numpy as np

# --- inputs you already have ---
# asset_df: TIME_SERIES_DAILY for SWDA.LON (Alpha Vantage), index=timestamp
# fx_df   : FX_DAILY for GBP->CHF (Alpha Vantage), index=timestamp
# Make sure you have: asset_df['asset_close_gbx'] and fx_df['close'] as before

asset_gbp = (asset_df['asset_close_gbx'] / 100.0)          # GBX -> GBP
fx_rate   = fx_df['close'].rename('GBPCHF')                 # GBP/CHF

# Align dates then compute returns
merged = pd.concat([asset_gbp, fx_rate], axis=1).dropna()
ret_asset = np.log(merged['asset_close_gbx']/100.0).diff().dropna()  # or np.log(merged['asset_close_gbx']/100.0).diff()
ret_fx    = np.log(merged['GBPCHF']).diff().dropna()

# Use common dates
df = pd.concat([ret_asset, ret_fx], axis=1).dropna()
df.columns = ['r_asset','r_fx']

# Annualized vols (252)
vol_asset = df['r_asset'].std() * np.sqrt(252)
vol_fx    = df['r_fx'].std()    * np.sqrt(252)
rho       = df['r_asset'].corr(df['r_fx'])

# Theoretical combined (annualized)
vol_comb_theory = np.sqrt(vol_asset**2 + vol_fx**2 + 2*rho*vol_asset*vol_fx)

# Actual CHF series (convert price then compute returns)
price_chf = (merged['asset_close_gbx']/100.0) * merged['GBPCHF']
ret_chf   = np.log(price_chf).diff().dropna()
vol_chf_actual = ret_chf.std() * np.sqrt(252)

print(f"SWDA vol (GBP):  {vol_asset:.2%}")
print(f"GBPCHF vol:      {vol_fx:.2%}")
print(f"Corr(asset, FX): {rho:.2f}")
print(f"Combined (theory): {vol_comb_theory:.2%}")
print(f"Combined (actual): {vol_chf_actual:.2%}")


In [ ]:
# GET DATA ON SPECIFIC DATE
symbol = ['ULVR.LON']

print(f"Processing {symbol}...")
url = (
    f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY'
    f'&symbol={symbol}&apikey={api_key}&outputsize=full&datatype=csv'
)
response = requests.get(url)
print(response.text[:500])  
# Print first 500 characters to see what is returned

# Try to parse only if 'timestamp' is in the header
if 'timestamp' in response.text.split('\n')[0]:
    df = pd.read_csv(StringIO(response.text), parse_dates=['timestamp'])
    df = df.sort_values('timestamp')  # oldest to newest

    # get price on specific date
    specific_date = '2024-10-08'
    if specific_date in df['timestamp'].values:
        price_on_date = df.loc[df['timestamp'] == specific_date, 'close'].values[0]
        print(f"Price on {specific_date}: {price_on_date}")
    else:
        print(f"No data available for {specific_date}")

    df_last_year = df.tail(252)
    prices = df_last_year['close']

    # Convert GBX to GBP if needed
    prices_gbp = prices / 100.0
else:
    print("Response does not contain a valid CSV with 'timestamp'. Check API key, symbol, or rate limits.")


# df_last_year = df.tail(252)
# prices = df_last_year['close']

# # Convert GBX to GBP if needed
# prices_gbp = prices / 100.0

Processing ['ULVR.LON']...
{
    "Error Message": "Invalid API call. Please retry or visit the documentation (https://www.alphavantage.co/documentation/) for TIME_SERIES_DAILY."
}
Response does not contain a valid CSV with 'timestamp'. Check API key, symbol, or rate limits.


In [30]:
%reset -f
%whos

Interactive namespace is empty.
